# 02 — Shuttlecock YOLO11 Fine-Tuning (Colab T4)

Week 2 deliverable: fine-tune `yolo11n.pt` on ~400 self-labeled shuttlecock bounding boxes and measure mAP improvement over the pre-trained baseline.

**Target:** mAP@0.5 ≥ 0.30 (realistic for a single-frame detector on 10-30 px objects). Full rationale in `~/.claude/plans/resilient-zooming-anchor.md`.

## Prerequisites
1. Colab runtime: T4 GPU (free tier).
2. Label Studio export converted to YOLO format + 80/20 train/val split, uploaded to `/content/data/shuttle/` with the layout described in `configs/yolo_shuttle_data.yaml`.
3. Both `configs/yolo_shuttle.yaml` and `configs/yolo_shuttle_data.yaml` uploaded alongside.
4. W&B API key in `WANDB_API_KEY` env var (set via Colab secret).

In [ ]:
!pip install -q ultralytics wandb
import os, random, numpy as np, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
import wandb
wandb.login()
wandb.init(project='rallylens-shuttle', name='finetune_v1', config={'seed': SEED})

## Pre-trained baseline mAP (before fine-tuning)

Run the stock `yolo11n.pt` on the val split for a reference point. This is the number we'll quote in the README's 'improvement over baseline' table — even if it's zero, it's a fair comparison.

In [ ]:
from ultralytics import YOLO

baseline = YOLO('yolo11n.pt')
baseline_metrics = baseline.val(
    data='configs/yolo_shuttle_data.yaml',
    imgsz=1280,
    split='val',
)
print('baseline mAP50:', baseline_metrics.box.map50)
print('baseline mAP50-95:', baseline_metrics.box.map)

## Fine-tune

`cfg=configs/yolo_shuttle.yaml` supplies mosaic=0, seed=42, imgsz=1280, save_period=1 (epoch checkpoints for Colab session drops).

In [ ]:
model = YOLO('yolo11n.pt')
results = model.train(
    data='configs/yolo_shuttle_data.yaml',
    cfg='configs/yolo_shuttle.yaml',
    project='runs/shuttle',
    name='finetune_v1',
    exist_ok=True,
)

## Post-fine-tune validation + export

Copy `runs/shuttle/finetune_v1/weights/best.pt` to `models/shuttle_best.pt` in the repo so `rallylens.vision.shuttlecock_detector.resolve_weights()` picks it up automatically.

In [ ]:
finetuned = YOLO('runs/shuttle/finetune_v1/weights/best.pt')
ft_metrics = finetuned.val(
    data='configs/yolo_shuttle_data.yaml',
    imgsz=1280,
    split='val',
)
print('fine-tuned mAP50:', ft_metrics.box.map50)
print('fine-tuned mAP50-95:', ft_metrics.box.map)

import shutil, pathlib
dst = pathlib.Path('models'); dst.mkdir(exist_ok=True)
shutil.copy('runs/shuttle/finetune_v1/weights/best.pt', dst / 'shuttle_best.pt')
print('copied -> models/shuttle_best.pt')
wandb.finish()